# Konteks / Skenario
Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas dipindahkan ke PySpark, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, membaca data langsung dari HDFS.

# Menyiapkan Dataset
Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/angger/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/angger/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


# Instruksi Pengerjaan
Buat notebook baru Tugas4_[NPM]_[Nama Lengkap].ipynb, buat SparkSession baru, lalu kerjakan bagian A sampai E berikut — seluruhnya wajib menggunakan PySpark, bukan pandas, dan data wajib dibaca langsung dari HDFS (hdfs://localhost:9000/...), bukan dari berkas lokal.

# SparkSession 

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum as spark_sum

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas4_2505060020_Angger") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/09/10 09:04:50 WARN Utils: Your hostname, angger resolves to a loopback address: 127.0.1.1; using 10.90.69.6 instead (on interface wlp3s0)
26/09/10 09:04:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 09:04:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/10 09:04:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


**A. Membaca dan Eksplorasi Awal (bobot 15%)**

Baca dataset dari HDFS, tampilkan printSchema(), jumlah baris (count()), dan 10 baris pertama (show(10)).

In [3]:
# Membaca file CSV dari HDFS
hdfs_path = "hdfs://localhost:9000/user/angger/tugas4/transaksi_september_2026.csv"
df_tugas = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# Tampilkan skema data
df_tugas.printSchema()

# Tampilkan jumlah baris
print("Jumlah total baris:", df_tugas.count())

# Tampilkan 10 baris pertama
df_tugas.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|  

**B. Menangani Data Kosong (bobot 15%)**

Kolom rating memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan df.na.fill() atau df.na.drop() (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [4]:
# Hitung jumlah rating yang bernilai null/kosong
null_rating_count = df_tugas.filter(col("rating").isNull()).count()
print("Jumlah data kosong pada kolom rating:", null_rating_count)

# Mengisi nilai kosong (null) pada kolom rating dengan nilai default/rata-rata
# Di sini kita pilih menggunakan .fillna() / .fill() agar data transaksi lain tidak terbuang
df_clean = df_tugas.na.fill({"rating": 3.0})

# Verifikasi tidak ada lagi nilai null
print("Jumlah data kosong setelah ditangani:", df_clean.filter(col("rating").isNull()).count())

Jumlah data kosong pada kolom rating: 204
Jumlah data kosong setelah ditangani: 0


**C. Transformasi Data (bobot 20%)**

Tambahkan kolom total_pendapatan (unit_terjual x harga_satuan), lalu tambahkan kolom tier_transaksi yang bernilai "Besar" jika total_pendapatan > 500000, atau "Kecil" jika sebaliknya

In [5]:
# 1. Menambahkan kolom total_pendapatan
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Menambahkan kolom tier_transaksi menggunakan conditional (when - otherwise)
df_transformed = df_transformed.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

# Tampilkan sampel hasil transformasi
df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



**D. Analisis dengan GroupBy (bobot 30%)**

Jawablah dengan kode PySpark (bukan pandas):

1. Kategori apa yang memiliki total_pendapatan tertinggi?
2. Kota mana dengan jumlah transaksi tier "Besar" terbanyak?
3. Berapa rata-rata rating untuk masing-masing metode_pembayaran (data kosong sudah ditangani di bagian B)?

**Total Pendapatan tertinggi**

In [6]:
kategori_tertinggi = df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc())

kategori_tertinggi.show(1)

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row



**Transaksi tier besar terbanyak**

In [8]:
kota_tier_besar = df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc())

kota_tier_besar.show(1)

+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row



**AVG untuk masing-masing metode_pembayaran**

In [9]:
avg_rating = df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc())

avg_rating.show()

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|3.948207171314741|
|    Transfer Bank|3.932806324110672|
|         E-Wallet|            3.904|
|     Kartu Kredit|3.861788617886179|
+-----------------+-----------------+



**E. Menyimpan Hasil ke HDFS (Bobot 20%)**  
Instruksi: Simpan DataFrame hasil olahan Bagian C ke HDFS dalam format CSV baru dan verifikasi keberhasilannya.  

In [10]:
output_hdfs_path = "hdfs://localhost:9000/user/angger/tugas4/hasil_transaksi_september"

# Menyimpan ke HDFS dalam format CSV
df_transformed.write.csv(output_hdfs_path, header=True, mode="overwrite")

print("Berhasil disimpan ke HDFS!")

# Verifikasi isi folder output di HDFS
!hdfs dfs -ls /user/angger/tugas4/hasil_transaksi_september

Berhasil disimpan ke HDFS!
Found 2 items
-rw-r--r--   3 angger supergroup          0 2026-09-10 09:17 /user/angger/tugas4/hasil_transaksi_september/_SUCCESS
-rw-r--r--   3 angger supergroup      97296 2026-09-10 09:17 /user/angger/tugas4/hasil_transaksi_september/part-00000-c302015b-6bb6-467c-9342-6c8446b07ab5-c000.csv


# Jangan lupa untuk menutup SparkSession

In [11]:
spark.stop()
print("SparkSession berhasil ditutup.")

SparkSession berhasil ditutup.
